In [ ]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import pyspark.sql.functions as F

from pyspark.sql.functions import col
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType

import utils.data_processing_bronze_table

import utils.data_processing_silver_table_lms
import utils.data_processing_silver_table_attributes
import utils.data_processing_silver_table_financials
import utils.data_processing_silver_table_clickstream

#import importlib
#importlib.reload(utils.data_processing_silver_table_clickstream)

import utils.data_processing_gold_table


## set up pyspark session

In [ ]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

## set up config

In [ ]:
# set up config
snapshot_date_str = "2023-01-01"

start_date_str = "2023-01-01"
end_date_str = "2023-03-01"

In [ ]:
# generate list of dates to process
def generate_first_of_month_dates(start_date_str, end_date_str):
    # Convert the date strings to datetime objects
    start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
    end_date = datetime.strptime(end_date_str, "%Y-%m-%d")
    
    # List to store the first of month dates
    first_of_month_dates = []

    # Start from the first of the month of the start_date
    current_date = datetime(start_date.year, start_date.month, 1)

    while current_date <= end_date:
        # Append the date in yyyy-mm-dd format
        first_of_month_dates.append(current_date.strftime("%Y-%m-%d"))
        
        # Move to the first of the next month
        if current_date.month == 12:
            current_date = datetime(current_date.year + 1, 1, 1)
        else:
            current_date = datetime(current_date.year, current_date.month + 1, 1)

    return first_of_month_dates

dates_str_lst = generate_first_of_month_dates(start_date_str, end_date_str)
dates_str_lst

## Build Bronze Table

In [ ]:
# create bronze datalake
bronze_lms_directory = "datamart/bronze/lms/"

bronze_clickstream_directory = "datamart/bronze/feature_clickstream/"
bronze_attributes_directory = "datamart/bronze/features_attributes/"
bronze_financials_directory = "datamart/bronze/features_financials/"

if not os.path.exists(bronze_lms_directory):
    os.makedirs(bronze_lms_directory)

if not os.path.exists(bronze_clickstream_directory):
    os.makedirs(bronze_clickstream_directory)

if not os.path.exists(bronze_attributes_directory):
    os.makedirs(bronze_attributes_directory)

if not os.path.exists(bronze_financials_directory):
    os.makedirs(bronze_financials_directory)



In [ ]:
# define the bronze file

lms_file = "data/lms_loan_daily.csv"
clickstream_file = "data/feature_clickstream.csv"
attributes_file = "data/features_attributes.csv"
financials_file = "data/features_financials.csv"

In [ ]:
#reload lib after changing the code

import importlib
import utils.data_processing_bronze_table
importlib.reload(utils.data_processing_bronze_table)

In [ ]:
# run bronze backfill for lms_file
for date_str in dates_str_lst:
    utils.data_processing_bronze_table.process_bronze_table(date_str, lms_file, bronze_lms_directory, spark)


In [ ]:
# inspect output
utils.data_processing_bronze_table.process_bronze_table(date_str, lms_file, bronze_lms_directory, spark).toPandas()

In [ ]:
# run bronze backfill for feature files
for date_str in dates_str_lst:
    utils.data_processing_bronze_table.process_bronze_table(date_str, clickstream_file, bronze_clickstream_directory, spark)

In [ ]:
#Inspect output
utils.data_processing_bronze_table.process_bronze_table(date_str, clickstream_file, bronze_clickstream_directory, spark).toPandas()

In [ ]:
# run bronze backfill for feature attributes file
for date_str in dates_str_lst:
    utils.data_processing_bronze_table.process_bronze_table(date_str, attributes_file, bronze_attributes_directory, spark)

In [ ]:
# run bronze backfill for feature financials file
for date_str in dates_str_lst:
    utils.data_processing_bronze_table.process_bronze_table(date_str, financials_file, bronze_financials_directory, spark)

## Build Silver Table

In [40]:
# create silver datalake for lms
silver_lms_daily_directory = "datamart/silver/lms_loan_daily/"

if not os.path.exists(silver_lms_daily_directory):
    os.makedirs(silver_lms_daily_directory)

# create silver datalake for attributes
silver_attributes_daily_directory = "datamart/silver/features_attributes/"

if not os.path.exists(silver_attributes_daily_directory):
    os.makedirs(silver_attributes_daily_directory)

# create silver datalake for financials
silver_financials_daily_directory = "datamart/silver/features_financials/"

if not os.path.exists(silver_financials_daily_directory):
    os.makedirs(silver_financials_daily_directory)

# create silver datalake for clickstream
silver_clickstream_daily_directory = "datamart/silver/features_clickstream/"

if not os.path.exists(silver_clickstream_daily_directory):
    os.makedirs(silver_clickstream_daily_directory)


In [ ]:
# run silver backfill for lms
for date_str in dates_str_lst:
    utils.data_processing_silver_table_lms.process_silver_table(date_str, bronze_lms_directory, silver_lms_daily_directory, spark)


In [ ]:
utils.data_processing_silver_table_lms.process_silver_table(date_str, bronze_lms_directory, silver_loan_daily_directory, spark).toPandas()

In [ ]:
# run silver backfill for attributes
for date_str in dates_str_lst:
    utils.data_processing_silver_table_attributes.process_silver_table(date_str, bronze_attributes_directory, silver_attributes_daily_directory, spark)


In [ ]:
utils.data_processing_silver_table_attributes.process_silver_table(date_str, bronze_attributes_directory, silver_attributes_daily_directory, spark).toPandas()

In [51]:
import importlib
importlib.reload(utils.data_processing_silver_table_financials)
importlib.reload(utils.data_processing_gold_table)

<module 'utils.data_processing_gold_table' from '/app/utils/data_processing_gold_table.py'>

In [ ]:
# run silver backfill for financials
for date_str in dates_str_lst:
    utils.data_processing_silver_table_financials.process_silver_table(date_str, bronze_financials_directory, silver_financials_daily_directory, spark)


In [ ]:
utils.data_processing_silver_table_financials.process_silver_table("2023-01-01", bronze_financials_directory, silver_financials_daily_directory, spark).toPandas()

In [ ]:
# run silver backfill for clickstream
for date_str in dates_str_lst:
    utils.data_processing_silver_table_clickstream.process_silver_table(date_str, bronze_clickstream_directory, silver_clickstream_daily_directory, spark)


In [ ]:
utils.data_processing_silver_table_clickstream.process_silver_table(date_str, bronze_clickstream_directory, silver_clickstream_daily_directory, spark).toPandas()

## EDA on credit labels

In [ ]:
# set dpd label definition
dpd = 30

# Path to the folder containing CSV files
folder_path = silver_lms_daily_directory

# Read all CSV files into a single DataFrame
files_list = [folder_path+os.path.basename(f) for f in glob.glob(os.path.join(folder_path, '*'))]
df = spark.read.option("header", "true").parquet(*files_list)

# filter only completed loans
df = df.filter(col("loan_start_date") < datetime.strptime("2024-01-01", "%Y-%m-%d"))

# create dpd flag if more than dpd
df = df.withColumn("dpd_flag", F.when(col("dpd") >= dpd, 1).otherwise(0))

# actual bads 
actual_bads_df = df.filter(col("installment_num") == 10)

# prepare for analysis
# df = df.filter(col("installment_num") < 10)

# visualise bad rate
pdf = df.toPandas()

# Group by col_A and count occurrences in col_B
grouped = pdf.groupby('mob')['dpd_flag'].mean()

# Sort the index (x-axis) of the grouped DataFrame
grouped = grouped.sort_index()

# Plotting
grouped.plot(kind='line', marker='o')

plt.title('DPD: '+ str(dpd))
plt.xlabel('mob')
plt.ylabel('bad rate')
plt.grid(True)
plt.show()


In [ ]:
df.show()

## Build gold table for labels

In [ ]:
# create bronze datalake
gold_label_store_directory = "datamart/gold/label_store/"

if not os.path.exists(gold_label_store_directory):
    os.makedirs(gold_label_store_directory)

In [ ]:
importlib.reload(utils.data_processing_gold_table)

In [52]:
# run gold backfill
for date_str in dates_str_lst:
    utils.data_processing_gold_table.process_gold_table(date_str, silver_lms_daily_directory, silver_attributes_daily_directory, silver_financials_daily_directory,
                       silver_clickstream_daily_directory,
                       gold_label_store_directory, spark, dpd=30, mob=6)

loaded from: datamart/silver/lms_loan_daily/silver_loan_daily_2023_01_01.parquet row count: 530
loaded lms: 530 attributes: 11974 financials: 11974 clickstream: 8974
final row count: 530 (should equal the lms row count after the cutoff filter)


saved to: datamart/gold/label_store/gold_ft_store_loan_panel.parquet
loaded from: datamart/silver/lms_loan_daily/silver_loan_daily_2023_02_01.parquet row count: 1031
loaded lms: 1031 attributes: 11974 financials: 11974 clickstream: 8974
final row count: 1031 (should equal the lms row count after the cutoff filter)


saved to: datamart/gold/label_store/gold_ft_store_loan_panel.parquet
loaded from: datamart/silver/lms_loan_daily/silver_loan_daily_2023_03_01.parquet row count: 1537
loaded lms: 1537 attributes: 11974 financials: 11974 clickstream: 8974
final row count: 1537 (should equal the lms row count after the cutoff filter)
saved to: datamart/gold/label_store/gold_ft_store_loan_panel.parquet


In [53]:
utils.data_processing_gold_table.process_gold_table('2023-01-01', silver_lms_daily_directory, gold_label_store_directory, spark, dpd = 30, mob = 6).toPandas()

TypeError: process_gold_table() missing 3 required positional arguments: 'silver_clickstream_daily_directory', 'gold_feature_store_directory', and 'spark'

In [47]:
utils.data_processing_gold_table.process_gold_table(date_str, silver_lms_daily_directory, silver_attributes_daily_directory, silver_financials_daily_directory,
                       silver_clickstream_daily_directory, gold_label_store_directory, spark, dpd = 30, mob = 6).dtypes


loaded from: datamart/silver/lms_loan_daily/silver_loan_daily_2023_03_01.parquet row count: 1537
loaded lms: 1537 attributes: 11974 financials: 11974 clickstream: 8974
final row count: 1537 (should equal the lms row count after the cutoff filter)


saved to: datamart/gold/label_store/gold_ft_store_loan_panel.parquet


[('Customer_ID', 'string'),
 ('loan_id', 'string'),
 ('loan_start_date', 'date'),
 ('tenure', 'int'),
 ('installment_num', 'int'),
 ('loan_amt', 'float'),
 ('due_amt', 'float'),
 ('paid_amt', 'float'),
 ('overdue_amt', 'float'),
 ('balance', 'float'),
 ('snapshot_date', 'date'),
 ('mob', 'int'),
 ('installments_missed', 'int'),
 ('first_missed_date', 'date'),
 ('dpd', 'int'),
 ('Age', 'int'),
 ('Occupation', 'string'),
 ('Annual_Income', 'decimal(10,2)'),
 ('Monthly_Inhand_Salary', 'decimal(10,2)'),
 ('Num_Bank_Accounts', 'int'),
 ('Num_Credit_Card', 'int'),
 ('Interest_Rate', 'int'),
 ('Num_of_Loan', 'int'),
 ('Type_of_Loan', 'string'),
 ('Delay_from_due_date', 'int'),
 ('Num_of_Delayed_Payment', 'int'),
 ('Changed_Credit_Limit', 'decimal(10,2)'),
 ('Num_Credit_Inquiries', 'int'),
 ('Credit_Mix', 'string'),
 ('Outstanding_Debt', 'decimal(10,2)'),
 ('Credit_Utilization_Ratio', 'decimal(10,2)'),
 ('Credit_History_Age', 'string'),
 ('Payment_of_Min_Amount', 'string'),
 ('Total_EMI_per_mo

In [56]:
utils.data_processing_gold_table.process_gold_table("2023-02-01", silver_lms_daily_directory, silver_attributes_daily_directory, silver_financials_daily_directory,
                       silver_clickstream_daily_directory, gold_label_store_directory, spark, dpd = 30, mob = 6).toPandas()

loaded from: datamart/silver/lms_loan_daily/silver_loan_daily_2023_02_01.parquet row count: 1031
loaded lms: 1031 attributes: 11974 financials: 11974 clickstream: 8974
final row count: 1031 (should equal the lms row count after the cutoff filter)
saved to: datamart/gold/label_store/gold_ft_store_loan_panel.parquet


,Customer_ID,snapshot_date,loan_id,loan_start_date,tenure,installment_num,loan_amt,due_amt,paid_amt,overdue_amt,...,fe_11,fe_12,fe_13,fe_14,fe_15,fe_16,fe_17,fe_18,fe_19,fe_20
0,CUS_0x1037,2023-02-01,CUS_0x1037_2023_01_01,2023-01-01,10,1,10000.0,1000.0,1000.0,0.0,...,100,183,39,179,161,71,208,-7,129,206
1,CUS_0x1069,2023-02-01,CUS_0x1069_2023_01_01,2023-01-01,10,1,10000.0,1000.0,1000.0,0.0,...,-82,284,57,146,120,67,-4,139,-26,-51
2,CUS_0x114a,2023-02-01,CUS_0x114a_2023_01_01,2023-01-01,10,1,10000.0,1000.0,1000.0,0.0,...,44,-17,118,-55,-47,93,50,173,50,274
3,CUS_0x1184,2023-02-01,CUS_0x1184_2023_01_01,2023-01-01,10,1,10000.0,1000.0,1000.0,0.0,...,131,123,157,56,267,127,190,8,20,214
4,CUS_0x1297,2023-02-01,CUS_0x1297_2023_01_01,2023-01-01,10,1,10000.0,1000.0,1000.0,0.0,...,182,141,244,318,215,37,29,172,266,59
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1026,CUS_0xd7c,2023-02-01,CUS_0xd7c_2023_02_01,2023-02-01,10,0,10000.0,0.0,0.0,0.0,...,121,72,70,151,181,153,80,65,-110,183
1027,CUS_0xdbf,2023-02-01,CUS_0xdbf_2023_02_01,2023-02-01,10,0,10000.0,0.0,0.0,0.0,...,222,112,77,125,172,-130,-17,162,57,84
1028,CUS_0xdea,2023-02-01,CUS_0xdea_2023_02_01,2023-02-01,10,0,10000.0,0.0,0.0,0.0,...,40,110,130,212,204,245,139,-1,197,179
1029,CUS_0xec8,2023-02-01,CUS_0xec8_2023_02_01,2023-02-01,10,0,10000.0,0.0,0.0,0.0,...,6,239,118,132,123,121,-29,176,126,9


## inspect label store

In [49]:
folder_path = gold_label_store_directory
files_list = [folder_path+os.path.basename(f) for f in glob.glob(os.path.join(folder_path, '*'))]
df = spark.read.option("header", "true").parquet(*files_list)
print("row_count:",df.count())

df.show()

row_count: 1537
+-----------+--------------------+---------------+------+---------------+--------+-------+--------+-----------+-------+-------------+---+-------------------+-----------------+---+---+-------------+-------------+---------------------+-----------------+---------------+-------------+-----------+--------------------+-------------------+----------------------+--------------------+--------------------+----------+----------------+------------------------+--------------------+---------------------+-------------------+-----------------------+--------------------+---------------+-------------------------+-----------+---------------------+-------------------------+------------------+---------------+---------------+-------------+---------------+--------------+-----------+---------+-----------+---------------------+--------------------+--------------------+-----------+-------------+-----------------+
|Customer_ID|             loan_id|loan_start_date|tenure|installment_num|loan_amt|d

In [50]:
df.printSchema()

root
 |-- Customer_ID: string (nullable = true)
 |-- loan_id: string (nullable = true)
 |-- loan_start_date: date (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- installment_num: integer (nullable = true)
 |-- loan_amt: float (nullable = true)
 |-- due_amt: float (nullable = true)
 |-- paid_amt: float (nullable = true)
 |-- overdue_amt: float (nullable = true)
 |-- balance: float (nullable = true)
 |-- snapshot_date: date (nullable = true)
 |-- mob: integer (nullable = true)
 |-- installments_missed: integer (nullable = true)
 |-- first_missed_date: date (nullable = true)
 |-- dpd: integer (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Occupation: string (nullable = true)
 |-- Annual_Income: decimal(10,2) (nullable = true)
 |-- Monthly_Inhand_Salary: decimal(10,2) (nullable = true)
 |-- Num_Bank_Accounts: integer (nullable = true)
 |-- Num_Credit_Card: integer (nullable = true)
 |-- Interest_Rate: integer (nullable = true)
 |-- Num_of_Loan: integer (nullable =